In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime
import gradio as gr

c:\Users\sesha\Desktop\ML Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [3]:
today = datetime.today()

system_message = f""" 
You are a smart friendly personal finance assistant that communicates entirely through natural conversation.
Your goal is to help users manage their finances simply and fundamentally.Respond in a clear, keeping the tone helpful and simple.
if your asks something not related to the finance tracking,
politely say you are only trained to answer questions about the finance tracking and its services. 
today's date is {today}.
"""

In [4]:
import sqlite3

DB = "expense_tracker.db"

conn = sqlite3.connect(DB)
cursor = conn.cursor()
conn.close()

In [5]:
def query(sql, params = None):
    conn = sqlite3.connect(DB)

    try:
        cursor = conn.cursor()
        cursor.execute(sql, params or [])
        conn.commit()
        result = cursor.fetchall()
        return result
    finally:
        conn.close()

In [6]:
sql = """
    CREATE TABLE IF NOT EXISTS salary (
        id INTEGER PRIMARY KEY,
        amount REAL,
        month TEXT,
        year INTEGER,
        created_at TEXT
    )
"""
query(sql)

[]

In [7]:
sql = """
    CREATE TABLE IF NOT EXISTS expenses (
        id INTEGER PRIMARY KEY,
        amount REAL,
        category TEXT,
        description TEXT,
        date TEXT,
        created_at TEXT
    )
"""
query(sql)

[]

In [8]:
def set_salary(amount, month, year):
    sql = "INSERT INTO salary (amount, month, year, created_at) VALUES (?, ?, ?, ?)"
    result = query(sql, [amount, month, year, datetime.now()])
    return result

In [9]:
def log_expense(amount, description, category, date):
    sql = "INSERT INTO expenses (amount, description, category, date, created_at) VALUES (?, ?, ?, ?, ?)"
    result = query(sql, [amount, description, category, date, datetime.now()])
    return result

In [10]:
def balance_function():
    yearmonth = today.strftime("%Y-%m-")   # e.g. "2026-04-"
    sql_salary = "SELECT ifnull(sum(amount), 0) FROM salary WHERE month = ? AND year = ?"
    sql_expenses = "SELECT ifnull(sum(amount), 0) FROM expenses WHERE date LIKE ?"
    result_salary = query(sql_salary, [today.strftime("%B"), today.year])
    result_expenses = query(sql_expenses, [yearmonth + "%"])   # "2026-04-%"
    balance = result_salary[0][0] - result_expenses[0][0]
    return balance

In [11]:
def summary_function():
    yearmonth = today.strftime("%Y-%m")
    sql_salary = "SELECT ifnull(sum(amount), 0) FROM salary WHERE month = ? AND year = ?"
    result_salary = query(sql_salary, [today.strftime("%B"), today.year])
    salary = result_salary[0][0]

    sql_expenses = f"SELECT category,ifnull(sum(amount), 0),ifnull(sum(amount), 0)/{salary} AS percentage FROM expenses  GROUP BY category HAVING date LIKE ?"
    result_expenses = query(sql_expenses, [yearmonth + "%"])
    return result_expenses

    

In [12]:
set_salary_function = {
    "name": "set_salary",
    "description": "Save user's monthly salary, wage or income, Store users income information as a numarical value for a specified period",
    "parameters": {
        "type": "object",
        "properties": {
            "amount": {
                "type": "number",
                "description": "Amount of total monthly salary or income"
            },
            "month": {
                "type": "string",
                "description": "Month for which the salary is being recorded"
            },
            "year": {
                "type": "number",
                "description": "Year for which the salary is being recorded"
            }
        },
        "required": ["amount","month","year"],
        "additionalProperties": False
    }
}

In [13]:
log_expense_function = {
    "name": "log_expense",
    "description": "Record an expense, Store users spending information",
    "parameters": {
        "type": "object",
        "properties": {
            "amount": {
                "type": "number",
                "description": "Amount spent"
            },
            "description": {
                "type": "string",
                "description": "Description of the expense"
            },
            "category": {
                "type": "string",
                "description": "Category of the expense"
            },
            "date": {
                "type": "string",
                "description": "Date for which the expense is being recorded"
            },
        },
        "required": ["amount", "description", "category", "date"],
        "additionalProperties": False
    }
}

In [14]:
get_balance_function = {
    "name": "get_balance",
    "description": "Get remaining balance, after subtracting total expenses from salary",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [15]:
get_expense_summary_function = {
    "name": "get_expense_summary",
    "description": "Get spending breakdown summary for a specified period ",
    "parameters": {
        "type": "object",
        "properties": {
            "month": {
                "type": "string",
                "description": "Month for which to get expense summary"
            },
            "year": {
                "type": "number",
                "description": "Year for which to get expense summary"
            }
        },
        "required": ["month", "year"],
        "additionalProperties": False
    }
}

In [16]:
tools = [ { "type": "function", "function": set_salary_function },
          { "type": "function", "function": log_expense_function },
          { "type": "function", "function": get_balance_function },
          { "type": "function", "function": get_expense_summary_function } 
        ]

In [17]:
def chat(message, history):

    messages = [{"role": "system", "content": system_message}]


    for h in history:
        # get all the previous conversation history
        messages.append({"role": h["role"], "content": h["content"]})
        # get user last question
    messages.append({"role": "user", "content": message})                                      
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
    return response.choices[0].message.content

In [18]:
def handle_tool_calls(message):
    response = []
    for tool_call in message.tool_calls:
        fcall = tool_call.function
        print(f"Processing tool call: {fcall.name}")
        arguments = json.loads(fcall.arguments)

        if fcall.name == "set_salary":
            amount = arguments.get("amount")
            month = arguments.get("month")
            year = arguments.get("year")
            res = set_salary(amount, month, year)

        elif fcall.name == "log_expense":
            amount = arguments.get("amount")
            description = arguments.get("description")
            category = arguments.get("category")
            date = arguments.get("date")
            res = log_expense(amount, description, category, date)

        elif fcall.name == "get_balance":
            res = balance_function()

        elif fcall.name == "get_expense_summary":
            res = summary_function()

        else:
            res = f"Unknown tool: {fcall.name}"

        response.append({
            "role": "tool",
            "content": str(res),
            "tool_call_id": tool_call.id
        })

    return response      

In [19]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [26]:
query("SELECT * FROM expenses")

[(1,
  12.0,
  'Food & Drink',
  'coffee',
  '2026-04-29',
  '2026-04-29 15:19:43.751205'),
 (2, 900.0, 'Housing', 'Rent', '2026-01-01', '2026-04-29 15:20:13.854393'),
 (3, 900.0, 'Housing', 'Rent', '2026-02-01', '2026-04-29 15:20:13.883964'),
 (4, 900.0, 'Housing', 'Rent', '2026-03-01', '2026-04-29 15:20:13.888968'),
 (5,
  15.99,
  'Subscriptions',
  'Netflix subscription renewal',
  '2026-04-01',
  '2026-04-29 15:20:58.626545'),
 (6,
  55.0,
  'Transportation',
  'Gas tank fill-up',
  '2026-04-01',
  '2026-04-29 15:20:58.650634'),
 (7,
  30.0,
  'Health',
  'Doctor appointment copay',
  '2026-04-29',
  '2026-04-29 15:20:58.658473')]

In [28]:
query("SELECT * FROM salary")

[(1, 2800.0, '04', 2026, '2026-04-29 15:18:02.585821'),
 (2, 3500.0, 'January', 2026, '2026-04-29 15:19:10.983335'),
 (3, 3500.0, 'February', 2026, '2026-04-29 15:19:11.014762'),
 (4, 3500.0, 'March', 2026, '2026-04-29 15:19:11.024291'),
 (5, 3500.0, 'april', 2026, '2026-04-29 15:23:14.691743'),
 (6, 4000.0, 'april', 2026, '2026-04-29 15:23:30.794246')]

In [22]:
query("DELETE FROM expenses")


[]

In [24]:
query("DELETE FROM salary")


[]